In [1]:
import pymysql

ip = "localhost"
port = "3306"
idd = "root"
pw = "<REDACTED>"

db = pymysql.connect(host=ip, user=idd, passwd=pw, charset='utf8')  # 커넥터 생성

cur = db.cursor()  # db 객체가 호출하는 cursor 생성

cmd = "show databases"   # DBMS에서는 반드시 마지막에 ";"/세미콜론을 붙여주어야 하지만 여기에선 생략한다.

cur.execute(cmd)   # 명령어 전송
rows = cur.fetchall()   # DBMS에게 commit을 전송하고 그 후에 반환되는 값을 rows에 초기화한다.

for i in rows:
    print(i)

cur.close()

('information_schema',)
('mysql',)
('performance_schema',)
('wp',)


In [ ]:
cmd = ["create database wp", "create user \'wp\'@\'172.16.25.2\' identified by \'<REDACTED>\'", "grant all privileges on wp.* to \'wp\'@\'172.16.25.2\'"]
for c in cmd:
    cur.execute(c)   # 명령어 전송
rows = cur.fetchall()   # DBMS에게 commit을 전송하고 그 후에 반환되는 값을 rows에 초기화한다.

for i in rows:
    print(i)

In [ ]:
import pymysql
import os

# 기본 연결 정보 (root로 초기 연결)
host_ip = input("호스트 IP(DBMS 서버 ip) (e.g., localhost): ")
port = int(input("port (특별한 경우가 아닌 한 3306 기본값): "))
root_id = "root"
root_pw = input("root password 입력 (e.g., <REDACTED>): ")  # 보안을 위해 입력받음

# DBMS 연결 설정
db = pymysql.connect(host=host_ip, port=port, user=root_id, passwd=root_pw, charset='utf8')
# cursor 생성
cur = db.cursor()

# 현재 선택된 DB 이름 (초기 None)
current_db_name = None

# --- 연결 및 초기 설정 기능 ---
def create_db_and_user():
    global current_db_name
    db_name = input("DB 이름 (e.g., wp): ")
    user_name = input("user 이름 (e.g., wp): ")
    user_password = input("password 입력 (e.g., <REDACTED>): ")
    cmds = [
        "create database %s" % db_name,
        "create user '%s'@'localhost' identified by '%s'" % (user_name, user_password),
        "grant all privileges on %s.* to '%s'@'localhost'" % (db_name, user_name)
    ]
    for cmd in cmds:
        cur.execute(cmd)
        db.commit()
        print("Executed: %s" % cmd)
    current_db_name = db_name
    switch_to_db()

def list_databases():
    cur.execute("SHOW DATABASES")
    rows = cur.fetchall()
    print("Available Databases:")
    for row in rows:
        print(row[0])

def select_database():
    global current_db_name
    db_name = input("Enter DB name to select: ")
    current_db_name = db_name
    switch_to_db()

def switch_to_db():
    if current_db_name:
        cur.execute("use %s" % current_db_name)
        print(f"Switched to database: {current_db_name}")
    else:
        print("No database selected. Please select a database first.")

# --- 테이블 관리 기능 (생성, 삽입, 업데이트 등) ---
def create_table():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    table_name = input("Enter table name (e.g., user): ")
    table_cmd = "create table %s(id INT, name varchar(10), address varchar(20))" % table_name
    cur.execute(table_cmd)
    db.commit()
    print(f"Table '{table_name}' created in '{current_db_name}'.")

def insert_data():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    table_name = input("Enter table name (e.g., user): ")
    id_val = input("Enter id value (e.g., 1): ")
    name_val = input("Enter name value (e.g., kong): ")
    address_val = input("Enter address value (e.g., 1234): ")
    insert_cmd = "insert into %s(id, name, address) values(%s, '%s', '%s')" % (table_name, id_val, name_val, address_val)
    cur.execute(insert_cmd)
    db.commit()
    print("Data inserted.")

# --- 단순 조회 기능 ---
def select_data():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    table_name = input("Enter table name (e.g., user): ")
    select_cmd = "select id, name, address from %s" % table_name
    cur.execute(select_cmd)
    rows = cur.fetchall()
    print("Data:")
    for row in rows:
        print(row)

def info_tables_in_db():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    info_cmd1 = "SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '%s'" % current_db_name
    cur.execute(info_cmd1)
    rows = cur.fetchall()
    print("Tables in %s:" % current_db_name)
    for row in rows:
        print(row)

def info_table_details():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    table_name = input("Enter table name (e.g., user): ")
    info_cmd2 = "SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = '%s' AND TABLE_SCHEMA = '%s'" % (table_name, current_db_name)
    cur.execute(info_cmd2)
    rows = cur.fetchall()
    print("Table info for %s:" % table_name)
    for row in rows:
        print(row)

def info_columns_in_table():
    if not current_db_name:
        print("No database selected. Please select a database first.")
        return
    table_name = input("Enter table name (e.g., user): ")
    info_cmd3 = "SELECT * FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '%s' AND TABLE_SCHEMA = '%s'" % (table_name, current_db_name)
    cur.execute(info_cmd3)
    rows = cur.fetchall()
    print("Columns in %s:" % table_name)
    for row in rows:
        print(row)

def info_views():
    cur.execute("SELECT * FROM INFORMATION_SCHEMA.VIEWS")
    rows = cur.fetchall()
    print("Views:")
    for row in rows:
        print(row)

def info_table_counts_by_schema():
    cur.execute("SELECT TABLE_SCHEMA, COUNT(*) FROM INFORMATION_SCHEMA.TABLES GROUP BY TABLE_SCHEMA")
    rows = cur.fetchall()
    print("Table counts by schema:")
    for row in rows:
        print(row)

# --- 설정 변경 기능 ---
def update_wp_config():
    config_path = input("Enter path to wp-config.php (e.g., /home/wp/wordpress/wp-config.php): ")
    db_name_to_use = current_db_name if current_db_name else input("DB 이름 (e.g., wp): ")
    user_name = input("user 이름 (e.g., wp): ")
    user_password = input("password 입력 (e.g., <REDACTED>): ")
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            lines = f.readlines()
        with open(config_path, 'w') as f:
            for line in lines:
                if line.startswith("define('DB_NAME'"):
                    f.write("define('DB_NAME', '%s');\n" % db_name_to_use)
                elif line.startswith("define('DB_USER'"):
                    f.write("define('DB_USER', '%s');\n" % user_name)
                elif line.startswith("define('DB_PASSWORD'"):
                    f.write("define('DB_PASSWORD', '%s');\n" % user_password)
                elif line.startswith("define('DB_HOST'"):
                    f.write("define('DB_HOST', '%s');\n" % host_ip)
                else:
                    f.write(line)
        print("wp-config.php updated successfully.")
    else:
        print("File not found: %s" % config_path)

while True:
    print(f"\nCurrent Database: {current_db_name if current_db_name else 'None'}")
    
    print("\n--- 연결 및 초기 설정 ---")
    print("1: 데이터베이스 목록 보기")
    print("2: 데이터베이스 선택")
    print("3: DB 및 사용자 생성 (그리고 전환)")
    
    print("\n--- 테이블 관리 ---")
    print("4: 테이블 생성")
    print("5: 데이터 삽입")
    
    print("\n--- 단순 조회 ---")
    print("6: 데이터 선택")
    print("7: 정보 - DB 내 테이블")
    print("8: 정보 - 테이블 상세")
    print("9: 정보 - 테이블 내 컬럼")
    print("10: 정보 - 뷰")
    print("11: 정보 - 스키마별 테이블 수")
    
    print("\n--- 설정 변경 ---")
    print("12: wp-config.php 업데이트")
    
    print("\nq: 종료")
    choice = input("Enter choice: ")
    
    if choice == '1':
        list_databases()
    elif choice == '2':
        select_database()
    elif choice == '3':
        create_db_and_user()
    elif choice == '4':
        create_table()
    elif choice == '5':
        insert_data()
    elif choice == '6':
        select_data()
    elif choice == '7':
        info_tables_in_db()
    elif choice == '8':
        info_table_details()
    elif choice == '9':
        info_columns_in_table()
    elif choice == '10':
        info_views()
    elif choice == '11':
        info_table_counts_by_schema()
    elif choice == '12':
        update_wp_config()
    elif choice == 'q':
        break

# cursor 및 DB 연결 종료
cur.close()
db.close()

호스트 IP(DBMS 서버 ip) (e.g., localhost):  172.16.10.150
port (특별한 경우가 아닌 한 3306 기본값):  3306
root password 입력 (e.g., <REDACTED>):  <REDACTED>



Current Database: None

--- 연결 및 초기 설정 ---
1: 데이터베이스 목록 보기
2: 데이터베이스 선택
3: DB 및 사용자 생성 (그리고 전환)

--- 테이블 관리 ---
4: 테이블 생성
5: 데이터 삽입

--- 단순 조회 ---
6: 데이터 선택
7: 정보 - DB 내 테이블
8: 정보 - 테이블 상세
9: 정보 - 테이블 내 컬럼
10: 정보 - 뷰
11: 정보 - 스키마별 테이블 수

--- 설정 변경 ---
12: wp-config.php 업데이트

q: 종료


Enter choice:  1


Available Databases:
information_schema
mysql
performance_schema
psw
wp

Current Database: None

--- 연결 및 초기 설정 ---
1: 데이터베이스 목록 보기
2: 데이터베이스 선택
3: DB 및 사용자 생성 (그리고 전환)

--- 테이블 관리 ---
4: 테이블 생성
5: 데이터 삽입

--- 단순 조회 ---
6: 데이터 선택
7: 정보 - DB 내 테이블
8: 정보 - 테이블 상세
9: 정보 - 테이블 내 컬럼
10: 정보 - 뷰
11: 정보 - 스키마별 테이블 수

--- 설정 변경 ---
12: wp-config.php 업데이트

q: 종료


Enter choice:  2
Enter DB name to select:  wp


...

Enter choice:  7


Tables in wp:
('def', 'wp', 'user', 'BASE TABLE', 'InnoDB', 10, 'Dynamic', 0, 0, 16384, 0, 0, 0, None, datetime.datetime(2025, 11, 10, 15, 14, 53), None, None, 'latin1_swedish_ci', None, '', '', 0, 'N')

Current Database: wp

--- 연결 및 초기 설정 ---
1: 데이터베이스 목록 보기
2: 데이터베이스 선택
3: DB 및 사용자 생성 (그리고 전환)

--- 테이블 관리 ---
4: 테이블 생성
5: 데이터 삽입

--- 단순 조회 ---
6: 데이터 선택
7: 정보 - DB 내 테이블
8: 정보 - 테이블 상세
9: 정보 - 테이블 내 컬럼
10: 정보 - 뷰
11: 정보 - 스키마별 테이블 수

--- 설정 변경 ---
12: wp-config.php 업데이트

q: 종료


Enter choice:  6
Enter table name (e.g., user):  user


Data:
(1, 'park', '1234')

Current Database: wp

--- 연결 및 초기 설정 ---
1: 데이터베이스 목록 보기
2: 데이터베이스 선택
3: DB 및 사용자 생성 (그리고 전환)

--- 테이블 관리 ---
4: 테이블 생성
5: 데이터 삽입

--- 단순 조회 ---
6: 데이터 선택
7: 정보 - DB 내 테이블
8: 정보 - 테이블 상세
9: 정보 - 테이블 내 컬럼
10: 정보 - 뷰
11: 정보 - 스키마별 테이블 수

--- 설정 변경 ---
12: wp-config.php 업데이트

q: 종료


Enter choice:  8
